# Swin-Tiny and DINOv2 Results Summary

This notebook summarises the transformer experiments for the CSC3109 aerial scene classification project.

**Task:** classify aerial images into four visually similar transport-infrastructure classes: `bridge`, `freeway`, `overpass`, and `railway`.

**Compared approaches:**

1. **DINOv2 ViT-S/14 linear probe** — frozen self-supervised DINOv2 backbone with only the classifier head trained.
2. **DINOv2 ViT-S/14 full fine-tuning** — all DINOv2 parameters fine-tuned on the project dataset.
3. **Swin-Tiny full fine-tuning** — ImageNet-pretrained hierarchical shifted-window transformer fine-tuned on the project dataset.


## Experiment protocol

- Dataset location: `data/raw/`.
- Training split: `data/raw/train`, 700 images per class, 2,800 images total.
- Held-out validation split: `data/raw/val`, 100 images per class, 400 images total.
- Training/tuning protocol: each training run used a stratified 80/20 internal split from `data/raw/train`.
- Final evaluation: the held-out validation set was used only after model training.
- Metrics reported: accuracy, macro precision, macro recall, macro F1, and confusion matrix.
- Input size: 224×224 RGB images with ImageNet-style normalization.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "uv.lock").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate project root. Run this notebook from the project or notebooks folder.")


PROJECT_ROOT = find_project_root()

RUNS = {
    "DINOv2 linear probe": {
        "model": "vit_small_patch14_dinov2.lvd142m",
        "approach": "Frozen backbone + classifier head",
        "metrics_path": PROJECT_ROOT / "reports" / "vit_small_patch14_dinov2_lvd142m_linear_probe_eval" / "metrics.json",
        "tune_metrics_path": PROJECT_ROOT / "model" / "vit_small_patch14_dinov2_lvd142m_linear_probe" / "best_tune_metrics.json",
        "checkpoint_path": PROJECT_ROOT / "model" / "vit_small_patch14_dinov2_lvd142m_linear_probe" / "best_model.pt",
        "parameters": "21.6M total, 1,540 trainable",
    },
    "DINOv2 fine-tune": {
        "model": "vit_small_patch14_dinov2.lvd142m",
        "approach": "Full model fine-tuning",
        "metrics_path": PROJECT_ROOT / "reports" / "vit_small_patch14_dinov2_lvd142m_eval" / "metrics.json",
        "tune_metrics_path": PROJECT_ROOT / "model" / "vit_small_patch14_dinov2_lvd142m_finetune" / "best_tune_metrics.json",
        "checkpoint_path": PROJECT_ROOT / "model" / "vit_small_patch14_dinov2_lvd142m_finetune" / "best_model.pt",
        "parameters": "21.6M total, 21.6M trainable",
    },
    "Swin-Tiny fine-tune": {
        "model": "swin_tiny_patch4_window7_224",
        "approach": "Full model fine-tuning",
        "metrics_path": PROJECT_ROOT / "reports" / "swin_tiny_eval" / "metrics.json",
        "tune_metrics_path": PROJECT_ROOT / "model" / "swin_tiny" / "best_tune_metrics.json",
        "checkpoint_path": PROJECT_ROOT / "model" / "swin_tiny" / "best_model.pt",
        "parameters": "27.5M total, 27.5M trainable",
    },
}

for run_name, config in RUNS.items():
    assert config["metrics_path"].exists(), f"Missing held-out metrics for {run_name}: {config['metrics_path']}"
    assert config["tune_metrics_path"].exists(), f"Missing tuning metrics for {run_name}: {config['tune_metrics_path']}"

print(f"Project root: {PROJECT_ROOT}")
print("All expected metric files found.")


In [ ]:
rows = []
all_metrics = {}
all_tune_metrics = {}

for run_name, config in RUNS.items():
    heldout = json.loads(config["metrics_path"].read_text())
    tune = json.loads(config["tune_metrics_path"].read_text())
    all_metrics[run_name] = heldout
    all_tune_metrics[run_name] = tune

    rows.append({
        "run": run_name,
        "model": config["model"],
        "approach": config["approach"],
        "parameters": config["parameters"],
        "tune_accuracy": tune["accuracy"],
        "tune_macro_f1": tune["macro_f1"],
        "heldout_accuracy": heldout["accuracy"],
        "heldout_macro_precision": heldout["macro_precision"],
        "heldout_macro_recall": heldout["macro_recall"],
        "heldout_macro_f1": heldout["macro_f1"],
        "heldout_errors": 400 - sum(heldout["confusion_matrix"][i][i] for i in range(4)),
    })

comparison = pd.DataFrame(rows).sort_values("heldout_macro_f1", ascending=False)
comparison


## Headline results

| Rank | Model / approach | Held-out accuracy | Held-out macro-F1 | Errors |
| ---: | --- | ---: | ---: | ---: |
| 1 | Swin-Tiny full fine-tuning | **100.00%** | **100.00%** | **0 / 400** |
| 2 | DINOv2 ViT-S/14 full fine-tuning | 99.25% | 99.25% | 3 / 400 |
| 3 | DINOv2 ViT-S/14 linear probe | 98.50% | 98.50% | 6 / 400 |

Swin-Tiny achieved the best held-out validation result in this run. DINOv2 fine-tuning substantially improved over the frozen-backbone DINOv2 linear probe, but it still made three validation errors.


In [ ]:
plot_df = comparison.melt(
    id_vars="run",
    value_vars=["heldout_accuracy", "heldout_macro_f1"],
    var_name="metric",
    value_name="score",
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="run", y="score", hue="metric")
plt.ylim(0.97, 1.005)
plt.title("Held-out validation performance")
plt.xlabel("")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Metric")
plt.tight_layout()
plt.show()


## Confusion matrices

The confusion matrices below use the class order:

`bridge`, `freeway`, `overpass`, `railway`


In [ ]:
class_names = ["bridge", "freeway", "overpass", "railway"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (run_name, metrics) in zip(axes, all_metrics.items()):
    confusion = pd.DataFrame(
        metrics["confusion_matrix"],
        index=[f"true {name}" for name in class_names],
        columns=[f"pred {name}" for name in class_names],
    )
    sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(run_name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

fig.tight_layout()
plt.show()


In [ ]:
per_class_rows = []
for run_name, metrics in all_metrics.items():
    report = metrics["classification_report"]
    for class_name in class_names:
        per_class_rows.append({
            "run": run_name,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1": report[class_name]["f1-score"],
            "support": report[class_name]["support"],
        })

per_class = pd.DataFrame(per_class_rows)
per_class


## Interpretation

### Swin-Tiny full fine-tuning

Swin-Tiny was the strongest final-model candidate in this comparison. It achieved **100.00% accuracy and 100.00% macro-F1** on the 400-image held-out validation set. Its confusion matrix had no mistakes, meaning all `bridge`, `freeway`, `overpass`, and `railway` validation images were classified correctly.

This result is consistent with the task structure. Swin's shifted-window attention and hierarchical feature extraction are well suited to aerial scene recognition because they combine local texture cues, such as road markings and rail patterns, with broader spatial layout cues, such as crossings, bridges, and surrounding land use.

### DINOv2 full fine-tuning

DINOv2 fine-tuning achieved **99.25% accuracy and 99.25% macro-F1** on the held-out validation set. It made three errors:

- 2 `bridge` images predicted as `overpass`
- 1 `freeway` image predicted as `overpass`

The bridge-to-overpass confusion is visually plausible because both categories can contain elevated transport structures, shadows, and similar linear road layouts. DINOv2's self-supervised features transferred very well, but in this run they were slightly less effective than Swin-Tiny for the fine-grained road-infrastructure classes.

### DINOv2 linear probe

The frozen-backbone DINOv2 linear probe achieved **98.50% accuracy and 98.50% macro-F1**. This is strong considering that only 1,540 classifier-head parameters were trained. It demonstrates that DINOv2 already provides useful generic visual representations. However, full fine-tuning was better, reducing held-out validation errors from 6 to 3.


## Report-ready summary

> We compared three transformer-based approaches for the four-class aerial scene classification task: a frozen-backbone DINOv2 ViT-S/14 linear probe, a fully fine-tuned DINOv2 ViT-S/14, and a fully fine-tuned Swin-Tiny Transformer. All models were trained using an internal stratified 80/20 split from the training set, while the official held-out validation set of 400 images was reserved for final evaluation. The DINOv2 linear probe achieved 98.50% accuracy and 98.50% macro-F1, showing that self-supervised foundation features transfer well to aerial imagery even with only the classifier head trained. Full DINOv2 fine-tuning improved performance to 99.25% accuracy and 99.25% macro-F1, with three remaining errors mostly involving confusion with the overpass class. Swin-Tiny achieved the best result, reaching 100.00% accuracy and 100.00% macro-F1 with no held-out validation errors. Based on this comparison, Swin-Tiny is selected as the recommended final model for deployment, while DINOv2 is retained as a strong foundation-model comparison approach.


## Generated artifacts

### DINOv2 linear probe

- Checkpoint: `model/vit_small_patch14_dinov2_lvd142m_linear_probe/best_model.pt`
- Training history: `model/vit_small_patch14_dinov2_lvd142m_linear_probe/history.csv`
- Held-out metrics: `reports/vit_small_patch14_dinov2_lvd142m_linear_probe_eval/metrics.json`
- Held-out confusion matrix: `reports/vit_small_patch14_dinov2_lvd142m_linear_probe_eval/confusion_matrix.png`

### DINOv2 full fine-tuning

- Checkpoint: `model/vit_small_patch14_dinov2_lvd142m_finetune/best_model.pt`
- Training history: `model/vit_small_patch14_dinov2_lvd142m_finetune/history.csv`
- Held-out metrics: `reports/vit_small_patch14_dinov2_lvd142m_eval/metrics.json`
- Held-out confusion matrix: `reports/vit_small_patch14_dinov2_lvd142m_eval/confusion_matrix.png`

### Swin-Tiny full fine-tuning

- Checkpoint: `model/swin_tiny/best_model.pt`
- Training history: `model/swin_tiny/history.csv`
- Held-out metrics: `reports/swin_tiny_eval/metrics.json`
- Held-out confusion matrix: `reports/swin_tiny_eval/confusion_matrix.png`
